# V4 Attribute Ranking — Domain × Tier Weighted Semantic Ranking

Ranks product attributes by combining:
1. **Embedding similarity** — cosine similarity via e5-large-v2
2. **Domain × Tier weight** — business-driven multiplier per domain/tier

Two modes:
- `"semantic"` — pure cosine similarity (no domain weight)
- `"weighted"` — cosine similarity × FINAL_WEIGHT (domain-driven)

**No sales weights** — importance is derived from domain/tier rules only.

Requires: `data_setup` notebook to have been run first.

In [0]:
# %pip install sentence-transformers scikit-learn --quiet

In [0]:
# dbutils.library.restartPython()

In [0]:
df = spark.read.parquet('dbfs:/FileStore/amogh/cdt/final_df')
display(df)

In [0]:
value_detail_df = spark.read.parquet(
    'dbfs:/FileStore/amogh/cdt/category_attribute_value_sales'
).toPandas()

In [0]:
import re

ABBREVIATION_MAP = {
    "STTD": "stated", "QLFD": "qualified", "PRSNC": "presence", "PRSNG": "presence",
    "CRN": "corn", "SGR": "sugar", "FAT": "fat", "CAL": "calorie", "PROT": "protein",
    "CRBHY": "carbohydrate", "SRVNG": "serving", "RNGS": "ranges",
    "FDA": "fda", "ORG": "organic", "MRKTNG": "marketing", "CLM": "claim",
    "EQ": "equivalent", "NUM": "number", "QTY": "quantity", "TTL": "total",
    "SBSTT": "substitute", "FLR": "flavor",
}

def expand_abbreviations(text: str) -> str:
    if text is None:
        return ""
    return " ".join(
        ABBREVIATION_MAP.get(
            re.sub(r"[^A-Z_]", "", w.upper()),
            w.lower()
        )
        for w in str(text).split()
    )

def clean_category_name(name: str) -> str:
    if not name:
        return ""
    name = re.sub(r"^\d+\s*-\s*", "", name)
    name = re.sub(
        r"\b(SG|GR|PP|MT|HB|NF|BK|FR|DD|FL|OB|RX|RW|EX|EW|ALL)\b",
        "",
        name,
        flags=re.I,
    )
    name = re.sub(r"[&/,\-]", " ", name)
    return re.sub(r"\s+", " ", name).strip().lower()

In [0]:
df = df.toPandas()

df["category"] = df["CATEGORY_NAME"].map(clean_category_name)
df["attribute"] = df["ATTRIBUTE_DESCRIPTION"].map(expand_abbreviations)

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Every CATEGORY_NAME is classified into exactly ONE domain using
# deterministic prefix + keyword rules (first-match-wins).
#
# DOMAINS (10 total):
#   FOOD_FRESH, FOOD_PACKAGED, FOOD_BEVERAGE, FOOD_SPECIALTY_DIETARY,
#   HEALTH_WELLNESS, HOUSEHOLD_UTILITY, HARDWARE_ELECTRONICS,
#   NONFOOD_DISCRETIONARY, NONFOOD_HOME, UNKNOWN
# ─────────────────────────────────────────────────────────────────────────

import pandas as pd
import re

def assign_domain(category: str) -> str:
    """
    Deterministic, rule-based domain classifier.
    Uses a tiered first-match-wins strategy:
      T0  Force UNKNOWN (junk / unclassifiable rows)
      T1  FOOD_SPECIALTY_DIETARY keyword overrides
      T2  HOUSEHOLD_UTILITY overrides within GR / SG / NF prefixes
      T3  HARDWARE_ELECTRONICS overrides within NF prefix + keywords
      T4  HEALTH_WELLNESS overrides within GR prefix
      T5  Pure prefix rules (PP, BK, MT, DD, ALL, EX/EW/RW, FR, GR, SG, HB, RX, FL, NF)
      T6  Keyword rules for unprefixed categories
      T7  Default → UNKNOWN
    """
    c = str(category).strip().strip('"')

    # ── TIER 0 ── Force UNKNOWN ──────────────────────────────────────────
    if re.search(r'UNCLASSIFIED|BAD CATEGORY|OBSOLETE|^Unclassified', c):
        return "UNKNOWN"

    # ── TIER 1 ── FOOD_SPECIALTY_DIETARY keyword overrides ───────────────
    if re.search(r'KOSHER', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"
    if re.search(r'HALAL', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"
    if re.search(r'ORGANIC', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"
    if re.search(r'MEATLESS', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"
    if re.search(r'PLANT.BASED', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"
    if re.search(r'ALL.NAT\b', c, re.I):
        return "FOOD_SPECIALTY_DIETARY"

    # ── TIER 1b ── FOOD_BEVERAGE keyword overrides ───────────────────────
    if re.search(
        r'SOFT DRINK|CARBONATED|\bSODA\b|\bJUICE\b|\bCOFFEE\b'
        r'|\bTEA\b|TEABAG|\bCOCOA\b|\bBEVERAGE|ALCOHOLIC'
        r'|(?<!PANCAKE )MILK(?!.*SUBSTITUTE)'
        r'|MILK SUB|INST MILK|DRINK MIX|HOT COCOA|\bDRINKS\b'
        r'|FLUID MILK|CANNED MILK', c, re.I
    ):
        if not re.search(r'WATERMELON|PANCAKE|FILTRATION|STEAK|^\d+-NF[\s_]', c, re.I):
            return "FOOD_BEVERAGE"

    # ── TIER 1c ── NONFOOD_HOME keyword overrides ────────────────────────
    if re.search(
        r'COOKWARE|BAKEWARE|TABLETOP|DOMESTICS|WHITE GOODS'
        r'|HOME DECOR|CLOSET (NEED|STORAGE)|NF BATH\b', c, re.I
    ):
        return "NONFOOD_HOME"

    # ── TIER 2 ── HOUSEHOLD_UTILITY overrides ────────────────────────────
    if re.search(r'^\d+-GR\s', c):
        if re.search(
            r'HOUSEHOLD CLEANING|DISH DETERG|BLEACH|FABRIC SOFT|LAUNDRY DETERG'
            r'|DISPOSABLE BAG|FOOD STORAGE|CUPS.PLATES|LITTER$', c
        ):
            return "HOUSEHOLD_UTILITY"
        if re.search(r'^\d+-GR STARCH$', c):
            return "HOUSEHOLD_UTILITY"
        if re.search(r'^\d+-GR PAPER$', c):
            return "HOUSEHOLD_UTILITY"

    if re.search(r'^\d+-SG\s', c):
        if re.search(r'HOUSEHOLD|LAUNDRY|PAPER PROD|HAND SOAP', c):
            return "HOUSEHOLD_UTILITY"

    if re.search(r'^\d+-NF[\s_]', c):
        if re.search(
            r'CLEANING|FOILWARE|HOUSEHOLD PLASTIC|RUBBER GLOVE|VACUUM BAG'
            r'|CANNING|WATER FILT|IRONING|SPONGE|PEST CONTROL|CLEANERS|STAIN', c
        ):
            return "HOUSEHOLD_UTILITY"

    # ── TIER 3 ── HARDWARE_ELECTRONICS overrides ─────────────────────────
    if re.search(r'^\d+-NF[\s_]', c):
        if re.search(r'APPLIANCE|BATTERIES|LIGHT BULB|HARDWARE|MOBILE ACC|AUTOMOTIVE', c):
            return "HARDWARE_ELECTRONICS"

    if re.search(r'ELECTRONIC WIRING', c):
        return "HARDWARE_ELECTRONICS"

    # ── TIER 4 ── HEALTH_WELLNESS override within GR ─────────────────────
    if re.search(r'^\d+-GR\s', c) and re.search(r'PERSONAL WASH', c):
        return "HEALTH_WELLNESS"

    # ── TIER 5 ── Pure prefix rules ──────────────────────────────────────
    if re.search(r'^\d+-PP\s', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-BK\s', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-MT\s', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-DD\s', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-ALL\s', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-(?:EX|EW|RW)\s', c):
        return "FOOD_FRESH"

    if re.search(r'^\d+-FR\s', c):
        return "FOOD_PACKAGED"
    if re.search(r'^\d+-GR\s', c):
        return "FOOD_PACKAGED"
    if re.search(r'^\d+-SG\s', c):
        return "FOOD_PACKAGED"

    if re.search(r'^\d+-HB\s', c):
        return "HEALTH_WELLNESS"
    if re.search(r'^\d+-RX\s', c):
        return "HEALTH_WELLNESS"

    if re.search(r'^\d+-FL\s', c):
        return "NONFOOD_DISCRETIONARY"
    if re.search(r'^\d+-NF[\s_]', c):
        return "NONFOOD_DISCRETIONARY"

    # ── TIER 6 ── Keyword rules for unprefixed categories ────────────────
    # FOOD_FRESH keywords
    if re.search(r'SALAMI|ITALIAN MEATS', c):
        return "FOOD_FRESH"
    if re.search(r'TURKEY BREAST', c):
        return "FOOD_FRESH"
    if re.search(r'ROAST BEEF|BEEF CUTS', c):
        return "FOOD_FRESH"
    if re.search(r'BOLOGNA|LIVERWURST', c):
        return "FOOD_FRESH"
    if re.search(r'HOT DOGS', c):
        return "FOOD_FRESH"
    if re.search(r'\bDELI\b', c):
        return "FOOD_FRESH"
    if re.search(r'SALAD BAR|PREPACK SALAD', c):
        return "FOOD_FRESH"
    if re.search(r'SUBS AND SANDWICH', c):
        return "FOOD_FRESH"
    if re.search(r'ENTREES AND SIDES', c):
        return "FOOD_FRESH"
    if re.search(r'OLIVE BAR|OLIVES.ANTI', c):
        return "FOOD_FRESH"
    if re.search(r'PORK AND RIBS', c):
        return "FOOD_FRESH"
    if re.search(r'\bSUSHI\b', c):
        return "FOOD_FRESH"
    if re.search(r'\bPIZZA\b', c):
        return "FOOD_FRESH"
    if re.search(r'FRESH PASTA', c):
        return "FOOD_FRESH"
    if re.search(r'\bHUMMUS\b', c):
        return "FOOD_FRESH"
    if re.search(r'SOUP', c):
        return "FOOD_FRESH"
    if re.search(r'SLICING CHEESE|SPECIALTY CHEESE|SPECIALTY BUTTER', c):
        return "FOOD_FRESH"
    if re.search(r'KNISHES|PIEROG', c):
        return "FOOD_FRESH"
    if re.search(r'BREAD & WRAPS', c):
        return "FOOD_FRESH"
    if re.search(r'GUAC|DIPS.*SALSA', c):
        return "FOOD_FRESH"
    if re.search(
        r'SMOKED FISH|STEAK FISH|CRABMEAT|SURIMI'
        r'|SPECIALTY SEAFOOD|FRESH PREPARED SEAFOOD', c
    ):
        return "FOOD_FRESH"
    if re.search(
        r'CHERRIES|PLANTAIN|POMEGRANATE|FRESH GREENS|BABY VEG|SNOW.*PEAS', c
    ):
        return "FOOD_FRESH"
    if re.search(r'PICKLES', c):
        return "FOOD_FRESH"
    if re.search(r'\bAPPY\b', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-(?:CHICKEN|HAM|TURKEY)$', c):
        return "FOOD_FRESH"
    if re.search(r'CREATIVE FRESH', c):
        return "FOOD_FRESH"
    if re.search(r'HOLIDAY.*DINNER', c):
        return "FOOD_FRESH"
    if re.search(r'\bGNG\b', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-CATERING$', c):
        return "FOOD_FRESH"
    if re.search(r'^\d+-DESSERTS$', c):
        return "FOOD_FRESH"
    if re.search(r'MISC FROZEN', c):
        return "FOOD_FRESH"

    # FOOD_PACKAGED keywords
    if re.search(r'NON-ALCOHOLIC BEVERAGES', c):
        return "FOOD_PACKAGED"
    if re.search(r'DRY GOODS', c):
        return "FOOD_PACKAGED"
    if re.search(r'\bSNACKING\b|\bSNACKS\b', c):
        return "FOOD_PACKAGED"
    if re.search(r'^\d+-BEVERAGES$', c):
        return "FOOD_PACKAGED"
    if re.search(r'INTERNATIONAL CUISINE', c):
        return "FOOD_PACKAGED"

    # HOUSEHOLD_UTILITY keywords
    if re.search(r'SUPPLIES/INGREDIENTS', c):
        return "HOUSEHOLD_UTILITY"
    if re.search(r'^\d+-SUPPLIES$', c):
        return "HOUSEHOLD_UTILITY"
    if re.search(r'SEAFOOD TOOLS', c):
        return "HOUSEHOLD_UTILITY"

    # NONFOOD_DISCRETIONARY keywords
    if re.search(r'GIFT CARDS', c):
        return "NONFOOD_DISCRETIONARY"
    if re.search(r'HOLIDAY GIFT', c):
        return "NONFOOD_DISCRETIONARY"
    if re.search(r'OPD TIP', c):
        return "NONFOOD_DISCRETIONARY"

    # HARDWARE_ELECTRONICS keywords
    if re.search(r'PAINT AND PAINT', c):
        return "HARDWARE_ELECTRONICS"

    # ── TIER 7 ── Default fallback ───────────────────────────────────────
    return "UNKNOWN"


# Apply to DataFrame
df["DOMAIN"] = df["CATEGORY_NAME"].apply(assign_domain)

In [0]:
display(df[["CATEGORY_NAME", "DOMAIN"]].drop_duplicates())

In [0]:
display(df[["DOMAIN"]].drop_duplicates())

In [0]:
display(df[["ATTRIBUTE_DESCRIPTION"]].drop_duplicates())

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# ATTRIBUTE BEHAVIOR TIERS  (inferred from ATTRIBUTE_DESCRIPTION keywords)
#
# TIERS (18 total):
#   OPER, META, BRAND, PACK, CLAIM, SRC, PI, BC,
#   FOOD, BEV, HW, HH, HPC, SENS_FOOD, SENS, TARG, NEUT, NUTR_PASSIVE
#
# NUTR_PASSIVE: Nutritional facts (e.g. sodium, potassium) that are
#   ubiquitous on labels but rarely drive purchase decisions.
# ─────────────────────────────────────────────────────────────────────────

import re

def classify_attribute(attr: str) -> str:
    """
    Classify an ATTRIBUTE_DESCRIPTION into a behavior tier
    using deterministic keyword / pattern rules only.
    """
    a = str(attr).strip().upper()

    # ── OPER: Operational identifiers / system noise ─────────────────
    if re.search(
        r'^WAKE_|^GTIN$|^UPC$|^PLU CODE$|^ITEM$|^OMNI ASIN$'
        r'|_MGR_NAME$|_STATUS_CODE|_FLAG$|_CODE$|_NUM$|_UOM$|_DESC$'
        r'|STOCK REORDER|MODEL NUMBER|^ITEM_', a
    ):
        return "OPER"

    # ── BRAND: Brand / label hierarchy ───────────────────────────────
    if re.search(
        r'^BRAND|PRIVATE.LABEL|LICENSED TRADEMARK'
        r'|^BRANDED VS|^LOW_BRAND', a
    ):
        return "BRAND"

    # ── META: Internal category / department groupings ───────────────
    if re.search(
        r'^CATEGORY$|^DEPARTMENT$|^DIVISION$|^MARKET$|^SEGMENT$'
        r'|^SUB CATEGORY$|^COMMODITY|^COMPETITIVE', a
    ):
        return "META"

    # ── SRC: Sourcing / provenance attributes ────────────────────────
    if re.search(
        r'^ORIGIN$|^GROWING AREA$|^IMPORTED OR DOMESTIC$'
        r'|^USDA GRADE$|^USDA ORGANIC SEAL$'
        r'|^SAFE HANDLING', a
    ):
        return "SRC"
    if re.search(
        r'^PI ORGANIC (CERTIFICATION|CRTFCTN|DEGREE)'
        r'|^PI GRASS FED|^PI FREE RANGE|^PI CAGE FREE'
        r'|^PI MADE IN USA', a
    ):
        return "SRC"

    # ── PI: Product Intelligence flags (nutritional / regulatory) ────
    if a.startswith("PI "):
        return "PI"

    # ── BC: Business Classification dimensions ───────────────────────
    if a.startswith("BC "):
        return "BC"

    # ── PACK: Packaging / size / serving / physical dimensions ───────
    if re.search(
        r'SERVING|^BASE SIZE$|^TOTAL SIZE$|^PRODUCT SIZE$'
        r'|^OUTER|^PACK TYPE$|^BONUS PACK$|^SECONDARY SIZE$'
        r'|PACKAGE|DIMENSION|^SHEET (COUNT|DIMENSION)'
        r'|^WEIGHT SIZE$|^POCKET COUNT$|^CELL SIZE$|^GRAIN SIZE$'
        r'|^CAKE SIZE$|^YIELD$|^YIELD CLAIM$|^WASHLOAD COUNT$', a
    ):
        return "PACK"

    # ── BEV: Beverage-specific product differentiators ───────────────
    if re.search(
        r'COFFEE (GRIND|AND TEA)|^ROAST TYPE$|^TEA TYPE$'
        r'|WINE|^ALCOHOL (BY VOLUME|PROOF)'
        r'|^COCKTAIL TYPE$|^JUICE PROCESS$'
        r'|^TOTAL PERCENT OF JUICE$|^PULP'
        r'|^MILK FAT$|^BEVERAGE COMPONENT$', a
    ):
        return "BEV"

    # ── HW: Hardware / electronics / appliance differentiators ───────
    if re.search(
        r'APPLIANCE|^WATTAGE$|^POWER SOURCE$|LIGHT OUTPUT|LUMENS'
        r'|^RAZOR|BRISTLE|^CELL SIZE$|^VIDEO FORMAT$|^CHEMICAL (NAME|SYSTEM)', a
    ):
        return "HW"

    # ── HH: Household / paper / cleaning differentiators ─────────────
    if re.search(
        r'PAPER PLY|PAPER COLOR|^ABSORBENCY|^COMPACTION'
        r'|INSECT TYPE|ODOR CONTROL|^STORAGE CONTAINER'
        r'|^REUSABILITY$|^SHEET COUNT$|^SHEET DIMENSION', a
    ):
        return "HH"

    # ── HPC: Health / personal care differentiators ──────────────────
    if re.search(
        r'HAIR |^TARGET GROUP (HAIR|SKIN|AGE|GENDER|CONDITION)'
        r'|^SUN PROTECTION|^DOSAGE$|^SUPPLEMENT TYPE$'
        r'|^FLUORIDE|^STRENGTH$|^STRENGTH CLAIM$'
        r'|SKIN CONDITION|^NIPPLE|^BOTTLE MATERIAL'
        r'|TARGET USE BOTTLE', a
    ):
        return "HPC"

    # ── FOOD: Food-specific product differentiators ──────────────────
    if re.search(
        r'MEAT (TYPE|VARIETY|CUT|COMPONENT|TOPPING)'
        r'|CHEESE (TYPE|COMPONENT)|BREAD TYPE|PASTA'
        r'|SEAFOOD TYPE|^FRUIT TYPE$|^VEGETABLE TYPE$'
        r'|^NUT TYPE$|^GRAIN TYPE$|^HERB TYPE$'
        r'|^OIL TYPE$|^OLIVE (TYPE|STUFFING)|^EGG INGREDIENT$'
        r'|^COOKIE TYPE$|^CRACKER TYPE$|^CEREAL TYPE$'
        r'|^DOUGHNUT TYPE$|^SNACK (BAR TYPE|COMPONENT)$'
        r'|^CAKE (TYPE|DECORATION|FLAVOR)'
        r'|^SAUSAGE VARIETY$|^CURD TYPE$|^CRUST TYPE$'
        r'|NOVELTY BASE|^PIZZA TOPPING'
        r'|^CHOCOLATE VARIETY$'
        r'|MANUFACTURING PROCESS|^SLICED OR UNSLICED$'
        r'|^NATURAL OR PROCESSED$|^ENTREE COURSE$'
        r'|^SOUP COURSE$|^SANDWICH (COURSE|COMPONENT|CRACKER|COOKIE)'
        r'|^POTATO COURSE$|^APPETIZER COURSE$|^CHILE COURSE$'
        r'|^DESSERT COMPONENT$'
        r'|^FRUIT OR FLAVOR'
        r'|^BIRD CLASSIFICATION$', a
    ):
        return "FOOD"

    # ── SENS_FOOD: Food-specific sensory (flavor, seasoning, etc.) ───
    if re.search(
        r'^FLAVOR$|^BASE FLAVOR$|ENROBING FLAVOR|FILLING FLAVOR'
        r'|FLAVOR SHARPNESS|^SWEETENER TYPE$|^SEASONING TYPE$'
        r'|^SAUCE (TYPE|AND SEASONING)|^TOPPING TYPE$', a
    ):
        return "SENS_FOOD"

    # ── SENS: Universal sensory / consumer-decision attributes ───────
    if re.search(
        r'^SCENT$|^COLOR$|^TEXTURE$|^FORM$|^FORMULATION$'
        r'|^THICKNESS$|^COMMON CONSUMER NAME$'
        r'|^NAKED PRODUCT (SOURCE|STYLE|SIZE)', a
    ):
        return "SENS"

    # ── TARG: Target / occasion / use-case attributes ────────────────
    if re.search(
        r'^TARGET (GROUP$|USE$|ANIMAL$)'
        r'|^OCCASION$|^APPLICATION AREA$|^FREQUENCY OF USE$'
        r'|^PREPARATION METHOD$|^PREPARATION METHOD ADDITIVE$'
        r'|^DURATION$|^DURATION CLAIM$|^SPORT$', a
    ):
        return "TARG"

    # ── CLAIM: General presence / regulatory claims ──────────────────
    if re.search(r'CLAIM$|STRATEGIC INGREDIENT', a):
        return "CLAIM"

    # ── Catch-alls ───────────────────────────────────────────────────
    if re.search(r'PRICE|PREPRICED|DEAL|FREE GOOD', a):
        return "OPER"
    if re.search(r'^IMPORTED', a):
        return "SRC"
    if re.search(r'^IMAGE|ENDORSEMENT|MANUFACTURER MARKETING|^TITLE$', a):
        return "BRAND"
    if re.search(r'^SODIUM$|^POTASSIUM$', a):
        return "NUTR_PASSIVE"
    if re.search(r'MANUFACTURER SUGGESTED|^IM MULTI', a):
        return "OPER"
    if re.search(r'PRODUCT STORAGE|ITEM NAME ON LABEL', a):
        return "PACK"
    if re.search(r'^CAPACITY$|^MATERIAL SUBSTANCE$', a):
        return "PACK"
    if re.search(
        r'ARTIFICIAL CLR|^VITAMIN TYPE$|^MINERAL TYPE$'
        r'|OIL AND FAT INGREDIENT|MARSHMALLOW PRESENCE', a
    ):
        return "FOOD"
    if re.search(r'^COVER COLOR$|^FASTENING METHOD$', a):
        return "HH"
    if re.search(r'^COMPONENT$|^TIP THICKNESS$|^FLOW SPEED$', a):
        return "SENS"
    if re.search(r'IH CBD|^RULE TYPE$|^RELEASE RATING$', a):
        return "OPER"

    return "NEUT"

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# DOMAIN → TIER PRIORITY MAPPING
#
# Priority levels map to fixed weights:
#   HIGH   = 1.0  — primary differentiator for this domain
#   MEDIUM = 0.7  — useful supporting signal
#   LOW    = 0.3  — noise or irrelevant in this domain
#
# Any tier NOT listed for a domain gets a DEFAULT weight of 0.5.
# ─────────────────────────────────────────────────────────────────────────

PRIORITY_WEIGHTS = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.3}
DEFAULT_WEIGHT = 0.5

DOMAIN_TIER_PRIORITY = {
    "FOOD_FRESH": {
        "HIGH":   {"FOOD", "SENS", "SENS_FOOD", "TARG", "SRC"},
        "MEDIUM": {"CLAIM", "PI", "BC", "NEUT"},
        "LOW":    {"OPER", "META", "BRAND", "PACK", "HW", "HH", "HPC", "BEV", "NUTR_PASSIVE"},
    },
    "FOOD_PACKAGED": {
        "HIGH":   {"FOOD", "SENS", "SENS_FOOD", "PACK"},
        "MEDIUM": {"CLAIM", "PI", "BC", "TARG", "NEUT", "BEV"},
        "LOW":    {"OPER", "META", "BRAND", "HW", "HH", "HPC", "SRC", "NUTR_PASSIVE"},
    },
    "FOOD_BEVERAGE": {
        "HIGH":   {"SENS", "SENS_FOOD", "BEV", "CLAIM"},
        "MEDIUM": {"FOOD", "PACK", "PI", "BC", "TARG", "NEUT", "SRC"},
        "LOW":    {"OPER", "META", "BRAND", "HW", "HH", "HPC", "NUTR_PASSIVE"},
    },
    "FOOD_SPECIALTY_DIETARY": {
        "HIGH":   {"PI", "CLAIM", "SENS", "SENS_FOOD", "SRC"},
        "MEDIUM": {"FOOD", "TARG", "BC", "NEUT"},
        "LOW":    {"OPER", "META", "BRAND", "PACK", "HW", "HH", "HPC", "BEV", "NUTR_PASSIVE"},
    },
    "HEALTH_WELLNESS": {
        "HIGH":   {"HPC", "TARG", "SENS"},
        "MEDIUM": {"CLAIM", "BC", "BRAND", "NEUT"},
        "LOW":    {"OPER", "META", "PACK", "FOOD", "SENS_FOOD", "PI", "HW", "HH", "BEV", "SRC", "NUTR_PASSIVE"},
    },
    "HOUSEHOLD_UTILITY": {
        "HIGH":   {"HH", "PACK", "SENS"},
        "MEDIUM": {"TARG", "BC", "CLAIM", "NEUT"},
        "LOW":    {"OPER", "META", "BRAND", "FOOD", "SENS_FOOD", "PI", "HPC", "HW", "BEV", "SRC", "NUTR_PASSIVE"},
    },
    "HARDWARE_ELECTRONICS": {
        "HIGH":   {"HW", "SENS"},
        "MEDIUM": {"PACK", "TARG", "BC", "HH", "NEUT"},
        "LOW":    {"OPER", "META", "BRAND", "FOOD", "SENS_FOOD", "PI", "HPC", "CLAIM", "BEV", "SRC", "NUTR_PASSIVE"},
    },
    "NONFOOD_DISCRETIONARY": {
        "HIGH":   {"TARG", "SENS"},
        "MEDIUM": {"BC", "BRAND", "HPC", "NEUT"},
        "LOW":    {"OPER", "META", "PACK", "FOOD", "SENS_FOOD", "PI", "HW", "HH", "CLAIM", "BEV", "SRC", "NUTR_PASSIVE"},
    },
    "NONFOOD_HOME": {
        "HIGH":   {"SENS", "PACK", "HH"},
        "MEDIUM": {"TARG", "BC", "BRAND", "NEUT"},
        "LOW":    {"OPER", "META", "FOOD", "SENS_FOOD", "PI", "HPC", "HW", "CLAIM", "BEV", "SRC", "NUTR_PASSIVE"},
    },
    "UNKNOWN": {
        "HIGH":   {"SENS"},
        "MEDIUM": {"BC", "TARG", "NEUT", "CLAIM", "FOOD", "SENS_FOOD"},
        "LOW":    {"OPER", "META", "BEV", "SRC", "NUTR_PASSIVE"},
    },
}

In [0]:
def assign_weight(domain: str, attribute_description: str) -> float:
    """
    Determine the FINAL_WEIGHT for an attribute in a domain context.

    Logic:
      1. Classify the attribute into a behavior tier.
      2. Look up the domain's priority mapping.
      3. Return HIGH (1.0), MEDIUM (0.7), LOW (0.3),
         or DEFAULT (0.5) if the tier is not listed.
    """
    tier = classify_attribute(attribute_description)
    priorities = DOMAIN_TIER_PRIORITY.get(domain, {})

    for level, tiers in priorities.items():
        if tier in tiers:
            return PRIORITY_WEIGHTS[level]

    return DEFAULT_WEIGHT


# Apply to DataFrame
df["FINAL_WEIGHT"] = df.apply(
    lambda row: assign_weight(row["DOMAIN"], row["ATTRIBUTE_DESCRIPTION"]),
    axis=1,
)

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# WHY:  Attributes with negligible sales or very low SKU coverage are
#       unlikely to be meaningful CDT differentiators.  Filtering them
#       early reduces noise and speeds up downstream processing.
#
# THRESHOLDS (conservative):
#   1. ATTRIBUTE_TOTAL_SPENT > 0        — remove zero-sales attributes
#   2. SKU coverage >= 5% of category   — attribute must appear on at
#      least 5% of SKUs in its category to be considered relevant
# ─────────────────────────────────────────────────────────────────────────

rows_before_prefilter = len(df)

# Sales filter: remove attributes with zero sales
df = df[df["ATTRIBUTE_TOTAL_SPENT"] > 0].copy()
rows_after_sales = len(df)

# SKU coverage filter: attribute must cover at least 5% of category SKUs
df["_SKU_COV"] = df["ATTRIBUTE_COVERAGE_SKU_COUNT"] / df["CATEGORY_SKU_COUNT"]
df = df[df["_SKU_COV"] >= 0.05].copy()
df.drop(columns=["_SKU_COV"], inplace=True)
rows_after_sku_cov = len(df)

print(f"=== Pre-Filter: Sales & SKU Coverage ===")
print(f"Rows before:              {rows_before_prefilter}")
print(f"After sales > 0:          {rows_after_sales}  (removed {rows_before_prefilter - rows_after_sales})")
print(f"After SKU coverage >= 5%: {rows_after_sku_cov}  (removed {rows_after_sales - rows_after_sku_cov})")
print(f"Total removed:            {rows_before_prefilter - rows_after_sku_cov}\n")

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# WHY:  Some attributes appear in nearly every category AND cover the
#       vast majority of SKUs within those categories.  These are
#       "always-on" attributes (e.g. system flags, universal metadata)
#       that carry zero discriminative value for CDT — they cannot
#       separate shopper choices because every product has the same
#       attribute present.
#
# HOW:  Two criteria must BOTH be true for an attribute to be excluded:
#   1. It appears in >85 % of all categories.
#   2. In >85 % of those categories it covers >80 % of SKUs.
#
# USES: Precomputed columns from data_setup:
#   ATTRIBUTE_COVERAGE_SKU_COUNT — distinct SKUs carrying this attribute
#   CATEGORY_SKU_COUNT           — total distinct SKUs in the category
# ─────────────────────────────────────────────────────────────────────────

total_categories = df["CATEGORY_INTERNAL_ID"].nunique()

# Criterion 1: number of categories each attribute appears in
attr_category_counts = (
    df.groupby("ATTRIBUTE_DESCRIPTION")["CATEGORY_INTERNAL_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CATEGORY_INTERNAL_ID": "category_count"})
)

# Criterion 2: in how many categories does the attribute cover >80 % of SKUs?
df["SKU_COVERAGE_FRAC"] = (
    df["ATTRIBUTE_COVERAGE_SKU_COUNT"] / df["CATEGORY_SKU_COUNT"]
)
majority_sku_df = df[df["SKU_COVERAGE_FRAC"] > 0.8].copy()
majority_counts = (
    majority_sku_df
    .groupby("ATTRIBUTE_DESCRIPTION")["CATEGORY_INTERNAL_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CATEGORY_INTERNAL_ID": "majority_category_count"})
)

# Merge and apply both criteria
merged = pd.merge(
    attr_category_counts, majority_counts,
    on="ATTRIBUTE_DESCRIPTION", how="left"
).fillna(0)

exclusion1_df = merged[
    (merged["category_count"] > total_categories * 0.85) &
    (merged["majority_category_count"] > total_categories * 0.85)
].copy()

excluded_universal = set(exclusion1_df["ATTRIBUTE_DESCRIPTION"])
df = df[~df["ATTRIBUTE_DESCRIPTION"].isin(excluded_universal)].copy()
df.drop(columns=["SKU_COVERAGE_FRAC"], inplace=True)

print(f"Total categories: {total_categories}")
print(f"\n=== Exclusion 1: Universal Attributes ===")
print(f"Excluded count: {len(excluded_universal)}")
print("Excluded attributes:")
for attr in sorted(excluded_universal):
    print(f"  • {attr}")
print(f"\nRows remaining: {len(df)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# WHY:  An attribute with only ONE distinct value in a category is a
#       constant — it cannot split the assortment into meaningful groups.
#       For example, if every SKU in "Frozen Pizza" has
#       PI KRILL OIL STTD = "No", that attribute adds nothing.
#
#       We exclude attributes that are single-valued in >80 % of the
#       categories they appear in.  A small percentage of categories
#       may have two values, but the attribute is overwhelmingly useless.
#
# HOW:  For each attribute, count categories where VALUE_DIVERSITY == 1.
#       If that count exceeds 80 % of the attribute's total categories,
#       exclude it.
# ─────────────────────────────────────────────────────────────────────────

# Per-attribute: how many categories does it appear in?
total_per_attr = (
    df.groupby("ATTRIBUTE_DESCRIPTION")["CATEGORY_INTERNAL_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CATEGORY_INTERNAL_ID": "total_categories"})
)

# Per-attribute: how many categories have only 1 distinct value?
single_val = df[df["VALUE_DIVERSITY"] == 1].copy()
single_val_counts = (
    single_val
    .groupby("ATTRIBUTE_DESCRIPTION")["CATEGORY_INTERNAL_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CATEGORY_INTERNAL_ID": "categories_with_1_value"})
)

eda1_merged = pd.merge(single_val_counts, total_per_attr, on="ATTRIBUTE_DESCRIPTION")
eda1_merged["pct_single_value"] = (
    eda1_merged["categories_with_1_value"] / eda1_merged["total_categories"] * 100
).round(1)

excluded_single_value = set(
    eda1_merged[eda1_merged["pct_single_value"] > 80]["ATTRIBUTE_DESCRIPTION"]
)

df = df[~df["ATTRIBUTE_DESCRIPTION"].isin(excluded_single_value)].copy()

print(f"\n=== Exclusion 2: Single-Value Attributes ===")
print(f"Excluded count: {len(excluded_single_value)}")
print("Excluded attributes:")
for attr in sorted(excluded_single_value):
    print(f"  • {attr}")
print(f"\nRows remaining: {len(df)}")



In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Combined summary of all exclusion steps.
# ─────────────────────────────────────────────────────────────────────────
all_excluded = excluded_universal | excluded_single_value

summary = pd.DataFrame({
    "Exclusion Step": [
        "1. Universal (>85% cats + >80% SKU coverage)",
        "2. Single-value (diversity=1 in >80% cats)",
        "TOTAL UNIQUE EXCLUDED",
    ],
    "Count": [
        len(excluded_universal),
        len(excluded_single_value),
        len(all_excluded),
    ]
})
display(summary)
print(f"\nFinal row count after all exclusions: {len(df)}")

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("intfloat/e5-large-v2")

In [0]:
# ============================================================
# Two modes:
#   "semantic"  → pure cosine similarity (embedding only)
#   "weighted"  → cosine similarity × domain-aware FINAL_WEIGHT
# ============================================================


def rank_attributes(
    df,
    category_internal_id: int,
    value_detail_df=None,
    mode: str = "weighted",
):
    """
    Rank attributes for a single category.

    Modes
    -----
    "semantic" : SCORE = cosine_similarity(attribute, category)
    "weighted" : SCORE = cosine_similarity(attribute, category) × FINAL_WEIGHT
    """

    # --- Filter to category -------------------------------------------
    cat_df = df[df["CATEGORY_INTERNAL_ID"] == category_internal_id].copy()
    if cat_df.empty:
        return cat_df

    # --- Embed clean attribute name only (no value enrichment) --------
    cat_df["embed_text"] = cat_df["attribute"]

    # --- Compute cosine similarity via embeddings ---------------------
    attr_texts = cat_df["embed_text"].tolist()
    category_text = cat_df["category"].iloc[0]

    attr_emb = embed_model.encode(attr_texts, normalize_embeddings=True)
    cat_emb = embed_model.encode(category_text, normalize_embeddings=True)

    cat_df["SEMANTIC_SCORE"] = np.dot(attr_emb, cat_emb)

    # --- Score --------------------------------------------------------
    if mode == "semantic":
        cat_df["SCORE"] = cat_df["SEMANTIC_SCORE"]
    elif mode == "weighted":
        cat_df["SCORE"] = cat_df["SEMANTIC_SCORE"] * cat_df["FINAL_WEIGHT"]
    else:
        raise ValueError("mode must be one of: 'semantic', 'weighted'")

    return (
        cat_df
        .sort_values("SCORE", ascending=False)
        .loc[:, [
            "CATEGORY_INTERNAL_ID",
            "CATEGORY_NAME",
            "ATTRIBUTE_ID",
            "ATTRIBUTE_DESCRIPTION",
            "ATTRIBUTE_COVERAGE_SKU_COUNT",
            "ATTRIBUTE_TOTAL_SPENT",
            "FINAL_WEIGHT",
            "SCORE"
        ]]
        .reset_index(drop=True)
    )

In [0]:
# ============================================================
# Returns the top-N ranked attributes along with their full
# value-level breakdown: every attribute value, its sales,
# and SKU count.
# ============================================================


def rank_attributes_detailed(
    df,
    category_internal_id: int,
    value_detail_df,
    mode: str = "weighted",
    top_n: int = 20,
):
    """
    Rank attributes for a category and enrich the top N with
    value-level detail (all attribute values, sales per value,
    SKU count per value).

    Returns
    -------
    ranked_df : DataFrame
        Top-N attributes with their rank and score.
    detail_df : DataFrame
        Value-level breakdown for those top-N attributes, sorted
        by rank then by value sales descending.
    """

    # --- Get ranked attributes ----------------------------------------
    ranked = rank_attributes(
        df,
        category_internal_id,
        value_detail_df=value_detail_df,
        mode=mode,
    )
    if ranked.empty:
        return ranked, pd.DataFrame()

    top_ranked = ranked.head(top_n).copy()
    top_ranked["RANK"] = range(1, len(top_ranked) + 1)

    # --- Filter value-level detail to this category & top attributes --
    top_attr_ids = set(top_ranked["ATTRIBUTE_ID"])

    detail = value_detail_df[
        (value_detail_df["CATEGORY_INTERNAL_ID"] == category_internal_id) &
        (value_detail_df["ATTRIBUTE_ID"].isin(top_attr_ids))
    ].copy()

    # --- Merge rank info into detail ----------------------------------
    detail = detail.merge(
        top_ranked[["ATTRIBUTE_ID", "RANK", "SCORE"]],
        on="ATTRIBUTE_ID",
        how="left"
    )

    detail = detail.sort_values(
        ["RANK", "ATTR_TOTAL_SPENT"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # --- Select final columns (no value-level spent) ------------------
    detail = detail[[
        "RANK",
        "ATTRIBUTE_DESCRIPTION",
        "ATTRIBUTE_VALUE_DESCRIPTION",
        "ATTR_SKU_COUNT",
        "SCORE"
    ]]

    return top_ranked, detail

In [0]:
display(rank_attributes(df, 879771, value_detail_df=value_detail_df, mode="weighted"))

In [0]:
display(rank_attributes(df, 879696, value_detail_df=value_detail_df, mode="weighted"))

In [0]:
display(rank_attributes(df, 879999, value_detail_df=value_detail_df, mode="weighted"))

In [0]:
display(rank_attributes(df, 880115, value_detail_df=value_detail_df, mode="weighted"))

In [0]:
display(rank_attributes(df, 880100, value_detail_df=value_detail_df, mode="weighted"))

In [0]:
top_ranked, detail = rank_attributes_detailed(df, 879999, value_detail_df, mode="weighted", top_n=20)
print("=== Top 20 Ranked Attributes ===")
display(top_ranked)
print("\n=== Value-Level Breakdown ===")
display(detail)

In [0]:
top_ranked, detail = rank_attributes_detailed(df, 880149, value_detail_df, mode="weighted", top_n=20)
print("=== Top 20 Ranked Attributes ===")
display(top_ranked)
print("\n=== Value-Level Breakdown ===")
display(detail)

In [0]:
top_ranked, detail = rank_attributes_detailed(df, 879696, value_detail_df, mode="weighted", top_n=20)
print("=== Top 20 Ranked Attributes ===")
display(top_ranked)
print("\n=== Value-Level Breakdown ===")
display(detail)

In [0]:
def export_top_n_for_llm(
    df,
    category_internal_id: int,
    value_detail_df,
    mode: str = "weighted",
    top_n: int = 50,
    top_n_values: int = 10,
):
    """
    Build an enriched top-N DataFrame suitable for LLM reranking.

    NOTE: RANK and SCORE are intentionally excluded to avoid biasing
    the LLM's independent judgment.

    Columns returned:
      CATEGORY_NAME, ATTRIBUTE_ID,
      ATTRIBUTE_DESCRIPTION, ATTRIBUTE_COVERAGE_SKU_COUNT,
      ATTRIBUTE_TOTAL_SPENT,
      TOP_VALUES (string: "value1 ($sales, n SKUs); value2 ...")
    """
    ranked = rank_attributes(
        df, category_internal_id, value_detail_df=value_detail_df, mode=mode
    )
    if ranked.empty:
        return ranked

    top_ranked = ranked.head(top_n).copy()

    # Build top values summary string per attribute
    top_attr_ids = set(top_ranked["ATTRIBUTE_ID"])
    detail = value_detail_df[
        (value_detail_df["CATEGORY_INTERNAL_ID"] == category_internal_id)
        & (value_detail_df["ATTRIBUTE_ID"].isin(top_attr_ids))
    ].copy()

    if not detail.empty:
        sales_col = "ATTR_TOTAL_SPENT" if "ATTR_TOTAL_SPENT" in detail.columns else "ATTRIBUTE_TOTAL_SPENT"
        detail[sales_col] = pd.to_numeric(detail[sales_col], errors="coerce").fillna(0)
        detail = detail.sort_values(
            ["ATTRIBUTE_ID", sales_col], ascending=[True, False]
        )
        top_vals = detail.groupby("ATTRIBUTE_ID").head(top_n_values)

        def summarize_values(group):
            parts = []
            for _, row in group.iterrows():
                val = str(row["ATTRIBUTE_VALUE_DESCRIPTION"]).strip()
                spent = row.get(sales_col, 0)
                skus = row.get("ATTR_SKU_COUNT", 0)
                parts.append(f"{val} (${spent:,.0f}, {int(skus)} SKUs)")
            return " ; ".join(parts)

        value_summary = (
            top_vals.groupby("ATTRIBUTE_ID")
            .apply(summarize_values)
            .reset_index()
            .rename(columns={0: "TOP_VALUES"})
        )
        top_ranked = top_ranked.merge(value_summary, on="ATTRIBUTE_ID", how="left")
    else:
        top_ranked["TOP_VALUES"] = ""

    top_ranked["TOP_VALUES"] = top_ranked["TOP_VALUES"].fillna("")
    top_ranked.drop(columns=["SCORE", "FINAL_WEIGHT", "CATEGORY_INTERNAL_ID"], inplace=True, errors="ignore")
    return top_ranked


In [0]:
# Change category_internal_id as needed
llm_input_df = export_top_n_for_llm(df, 879999, value_detail_df, mode="weighted", top_n=50)
display(llm_input_df)